# 파인튜닝 모델과 에이전트 통합

> 모델은 '뇌'이고 에이전트는 '몸'이다 -- 뇌를 바꾸면 행동이 바뀐다

Phase 4에서 파인튜닝한 모델은 단독으로는 '질문-응답 기계'에 불과하다.  
이 모델을 **에이전트의 추론 엔진**으로 장착하면, 스스로 계획하고 도구를 사용하며 결과를 기억하는 **자율 시스템**이 된다.

---

### 학습 목표

| # | 목표 |
|---|------|
| 1 | LLM Agent의 4대 요소(Planning, Tool Use, Memory, Action)를 설명할 수 있다 |
| 2 | Tool-use(Function Calling) 데이터 형식을 이해하고 학습 데이터를 구성할 수 있다 |
| 3 | LangChain의 핵심 컴포넌트(LLM, Prompt, Chain, Tool, Memory) 역할을 설명할 수 있다 |
| 4 | Ollama 모델을 LangChain에 연결하여 LCEL 체인을 구성할 수 있다 |
| 5 | 파인튜닝 모델을 에이전트 추론 엔진으로 사용했을 때의 이점을 정량적으로 설명할 수 있다 |

In [ ]:
# === 환경 설치 (필요 시 주석 해제) ===
# !pip install langchain langchain-ollama langchain-community requests -q

In [ ]:
import json
import sqlite3
import requests
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("Phase 10-01: 파인튜닝 모델과 에이전트 통합")
print(f"실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---
## 1. LLM 에이전트란 무엇인가

에이전트는 단순 질의응답을 넘어, **스스로 계획하고 도구를 사용하며 결과를 기억하는** 자율적 시스템이다.

### Agent 4대 요소

| 요소 | 역할 | 구현 예시 |
|------|------|----------|
| **Planning** | 복잡한 작업을 하위 단계로 분해 | Chain-of-Thought, ReAct 프롬프팅 |
| **Tool Use** | 외부 API/DB/파일 시스템 호출 | Function Calling, SQL 실행 |
| **Memory** | 대화 히스토리 및 중간 결과 저장 | 단기: 대화 버퍼, 장기: 벡터 DB |
| **Action** | 계획과 도구를 기반으로 실제 행동 수행 | 코드 실행, API 호출, 응답 생성 |

핵심: LLM이 **의사결정의 중심**에 있고, 나머지 요소가 LLM의 판단을 실현한다.

In [ ]:
# === Agent 아키텍처 시각화: 4요소 다이어그램 ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

# --- 박스 그리기 함수 ---
def draw_box(ax, xy, w, h, text, color, fontsize=11, fontcolor='white'):
    rect = plt.Rectangle(xy, w, h, facecolor=color, edgecolor='black', linewidth=2, zorder=2)
    ax.add_patch(rect)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=fontcolor, zorder=3)

def draw_arrow(ax, start, end, color='#333333'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', lw=2.5, color=color))

# --- User ---
draw_box(ax, (0.5, 4), 2, 1.5, 'User\n(질문/지시)', '#339af0')

# --- Agent (중앙) ---
draw_box(ax, (4, 3.5), 2.5, 2.5, 'Agent\n(LLM 기반\n의사결정)', '#ff6b6b')

# --- 4요소 ---
draw_box(ax, (7.5, 7.5), 2.2, 1.2, 'Planning\n작업 분해', '#845ef7')
draw_box(ax, (7.5, 5.5), 2.2, 1.2, 'Tool Use\n도구 호출', '#ffa94d', fontcolor='black')
draw_box(ax, (7.5, 3.5), 2.2, 1.2, 'Memory\n컨텍스트 유지', '#20c997')

# --- Action ---
draw_box(ax, (7.5, 1), 2.2, 1.2, 'Action\n실행 결정', '#51cf66')

# --- Environment ---
draw_box(ax, (11, 3.5), 2.5, 2.5, 'Environment\n(DB/API/\n외부시스템)', '#ffa94d', fontcolor='black')

# --- 화살표 ---
# User -> Agent
draw_arrow(ax, (2.5, 4.75), (4.0, 4.75))
# Agent -> 4요소
draw_arrow(ax, (6.5, 5.5), (7.5, 8.1))
draw_arrow(ax, (6.5, 4.75), (7.5, 6.1))
draw_arrow(ax, (6.5, 4.0), (7.5, 4.1))
# 4요소 -> Action
draw_arrow(ax, (8.6, 7.5), (8.6, 2.2), color='#845ef7')
draw_arrow(ax, (8.6, 5.5), (8.6, 2.2), color='#ffa94d')
draw_arrow(ax, (8.6, 3.5), (8.6, 2.2), color='#20c997')
# Action -> Environment
draw_arrow(ax, (9.7, 1.6), (11.0, 3.8))
# Environment -> Agent (Feedback loop)
draw_arrow(ax, (11.5, 6.0), (6.0, 6.0), color='#868e96')
ax.text(8.5, 6.3, '관찰 결과 / 피드백', ha='center', fontsize=9, color='#868e96', style='italic')

ax.set_title('LLM Agent 아키텍처: 4대 요소', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n에이전트 = LLM(뇌) + 계획 + 도구 + 기억 + 행동")
print("LLM이 바뀌면 → 계획 방식, 도구 선택, 행동 품질이 모두 바뀐다")

---
## 2. Tool-use 데이터 형식 (Function Calling)

파인튜닝 모델에게 도구 사용 능력을 가르치려면, **학습 데이터에 함수 호출 형식을 포함**해야 한다.

### Function Calling JSON 구조

```
User 질문 → Assistant가 function_call 생성 → 시스템이 실행 → 결과 반환 → Assistant가 해석
```

| 필드 | 역할 |
|------|------|
| `role: "assistant"` + `function_call` | 모델이 도구 호출을 결정 |
| `function_call.name` | 호출할 함수 이름 |
| `function_call.arguments` | 함수에 전달할 인자 (JSON 문자열) |
| `role: "function"` | 함수 실행 결과 반환 |

In [ ]:
# === Tool-use 데이터 예시 생성 ===

# 예시 1: SQL 실행
example_1 = {
    "messages": [
        {
            "role": "system",
            "content": "당신은 SQL 전문가입니다. 필요 시 execute_sql 함수를 호출하세요."
        },
        {
            "role": "user",
            "content": "직원 수가 가장 많은 부서는?"
        },
        {
            "role": "assistant",
            "content": None,
            "function_call": {
                "name": "execute_sql",
                "arguments": json.dumps({
                    "query": "SELECT department, COUNT(*) as cnt FROM employees GROUP BY department ORDER BY cnt DESC LIMIT 1"
                })
            }
        },
        {
            "role": "function",
            "name": "execute_sql",
            "content": json.dumps({"department": "Engineering", "cnt": 142})
        },
        {
            "role": "assistant",
            "content": "직원 수가 가장 많은 부서는 Engineering으로, 총 142명입니다."
        }
    ]
}

# 예시 2: 집계 함수 활용
example_2 = {
    "messages": [
        {
            "role": "system",
            "content": "당신은 SQL 전문가입니다. 필요 시 execute_sql 함수를 호출하세요."
        },
        {
            "role": "user",
            "content": "부서별 평균 급여를 높은 순으로 보여줘"
        },
        {
            "role": "assistant",
            "content": None,
            "function_call": {
                "name": "execute_sql",
                "arguments": json.dumps({
                    "query": "SELECT department, ROUND(AVG(salary), 0) as avg_salary FROM employees GROUP BY department ORDER BY avg_salary DESC"
                })
            }
        },
        {
            "role": "function",
            "name": "execute_sql",
            "content": json.dumps([
                {"department": "Engineering", "avg_salary": 95000},
                {"department": "Marketing", "avg_salary": 78000},
                {"department": "Sales", "avg_salary": 72000}
            ])
        },
        {
            "role": "assistant",
            "content": "부서별 평균 급여(높은 순):\n1. Engineering: 95,000원\n2. Marketing: 78,000원\n3. Sales: 72,000원"
        }
    ]
}

# 예시 3: 조건부 필터링
example_3 = {
    "messages": [
        {
            "role": "system",
            "content": "당신은 SQL 전문가입니다. 필요 시 execute_sql 함수를 호출하세요."
        },
        {
            "role": "user",
            "content": "급여가 100000 이상인 직원 중 입사일이 가장 최근인 사람은?"
        },
        {
            "role": "assistant",
            "content": None,
            "function_call": {
                "name": "execute_sql",
                "arguments": json.dumps({
                    "query": "SELECT name, salary, hire_date FROM employees WHERE salary >= 100000 ORDER BY hire_date DESC LIMIT 1"
                })
            }
        },
        {
            "role": "function",
            "name": "execute_sql",
            "content": json.dumps({"name": "김철수", "salary": 105000, "hire_date": "2024-11-15"})
        },
        {
            "role": "assistant",
            "content": "급여 100,000원 이상 직원 중 가장 최근 입사자는 김철수님으로, 급여 105,000원이며 2024년 11월 15일에 입사하셨습니다."
        }
    ]
}

# 출력
examples = [example_1, example_2, example_3]
for i, ex in enumerate(examples, 1):
    print(f"{'='*60}")
    print(f"  Tool-use 학습 데이터 예시 {i}")
    print(f"{'='*60}")
    for msg in ex["messages"]:
        role = msg["role"].upper()
        if "function_call" in msg:
            fc = msg["function_call"]
            print(f"  [{role}] -> function_call: {fc['name']}")
            print(f"            args: {fc['arguments']}")
        elif role == "FUNCTION":
            print(f"  [{role}] ({msg['name']}) -> {msg['content']}")
        else:
            content = msg["content"] or "(null - 함수 호출로 대체)"
            print(f"  [{role}] {content}")
    print()

print(f"학습 데이터 패턴: User 질문 -> Assistant function_call -> Function 결과 -> Assistant 해석")
print(f"이 패턴으로 파인튜닝하면 모델이 '언제 도구를 호출할지' 스스로 판단하게 된다")

---
## 3. LangChain 기초

LangChain은 LLM 애플리케이션을 구축하기 위한 프레임워크다.  
핵심 컴포넌트 5가지:

| 컴포넌트 | 역할 | 설명 |
|----------|------|------|
| **LLM** | 추론 엔진 | OpenAI, Ollama, HuggingFace 등 모델 래퍼 |
| **Prompt** | 입력 구성 | PromptTemplate, ChatPromptTemplate로 동적 프롬프트 생성 |
| **Chain** | 파이프라인 연결 | 여러 단계를 순차적으로 연결 (LCEL: `\|` 연산자) |
| **Tool** | 외부 도구 | SQL 실행, 웹 검색, 계산기 등 |
| **Memory** | 상태 관리 | ConversationBufferMemory, 대화 히스토리 유지 |

In [ ]:
# === LangChain 핵심 코드 구조 + 범용 vs 파인튜닝 비교 ===

print("=" * 60)
print("  LangChain 핵심 코드 구조 (LCEL 패턴)")
print("=" * 60)
print()

langchain_code = '''
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1) LLM: Ollama에 등록된 파인튜닝 모델
llm = ChatOllama(
    model="my-sql-model",
    base_url="http://localhost:11434",
    temperature=0.1,
)

# 2) Prompt: 동적 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "SQL 전문가. 스키마: {schema}"),
    ("human", "{question}"),
])

# 3) Chain: LCEL 파이프라인 ( | 연산자)
chain = prompt | llm | StrOutputParser()

# 4) 실행
result = chain.invoke({
    "schema": "employees(id, name, department, salary)",
    "question": "부서별 평균 급여는?"
})
'''
print(langchain_code)

print("\n핵심: prompt | llm | parser 세 줄이면 완전한 파이프라인이 완성된다")
print()

# --- 범용 LLM vs 파인튜닝 모델 비교 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ['단순 질문\n(인사/날씨)', '코드 생성\n(일반)', 'SQL 생성\n(도메인)', '스키마 해석\n(도메인)', '멀티턴 SQL\n(복합)']

# 정확도 데이터
general_acc = [92, 85, 62, 55, 48]
finetuned_acc = [88, 82, 91, 93, 87]

x = np.arange(len(categories))
width = 0.35

bars1 = axes[0].bar(x - width/2, general_acc, width, label='범용 LLM', color='#c4c4c4', edgecolor='black')
bars2 = axes[0].bar(x + width/2, finetuned_acc, width, label='파인튜닝 모델', color='#51cf66', edgecolor='black')
axes[0].set_ylabel('정확도 (%)')
axes[0].set_title('작업별 정확도 비교')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, fontsize=9)
axes[0].legend()
axes[0].set_ylim(0, 105)
axes[0].grid(axis='y', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=8)

# 응답 형식 일관성
consistency_labels = ['JSON 형식\n준수율', 'SQL 구문\n정확률', '한국어\n자연스러움', '에러 처리\n적절성']
general_con = [70, 65, 80, 55]
finetuned_con = [95, 93, 90, 88]

x2 = np.arange(len(consistency_labels))
bars3 = axes[1].bar(x2 - width/2, general_con, width, label='범용 LLM', color='#c4c4c4', edgecolor='black')
bars4 = axes[1].bar(x2 + width/2, finetuned_con, width, label='파인튜닝 모델', color='#51cf66', edgecolor='black')
axes[1].set_ylabel('일관성 (%)')
axes[1].set_title('출력 형식 일관성 비교')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(consistency_labels, fontsize=9)
axes[1].legend()
axes[1].set_ylim(0, 105)
axes[1].grid(axis='y', alpha=0.3)
for bar in bars3:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=8)
for bar in bars4:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=8)

plt.suptitle('범용 LLM vs 파인튜닝 모델: 에이전트 성능 영향', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n일반 작업에서는 범용 LLM이 충분하지만,")
print("도메인 특화 작업(SQL, 스키마)에서는 파인튜닝 모델이 압도적으로 우세하다")

---
## 4. Ollama + LangChain 연결

Phase 8에서 배포한 Ollama 모델을 LangChain 에이전트의 **두뇌**로 연결한다.

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Phase 4에서 파인튜닝한 모델을 Ollama로 로드
llm = ChatOllama(
    model="my-sql-model",       # Ollama에 등록한 모델명
    base_url="http://localhost:11434",
    temperature=0.1,             # SQL 생성이므로 낮은 temperature
)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 SQL 전문가입니다. 주어진 스키마를 참고하여 SQL을 생성하세요.\n스키마: {schema}"),
    ("human", "{question}"),
])

# LCEL 체인 구성
chain = prompt | llm | StrOutputParser()

# 실행
result = chain.invoke({
    "schema": "employees(id, name, department, salary)",
    "question": "부서별 평균 급여는?"
})
```

In [ ]:
# === Ollama 연결 실제 코드 + 연결 테스트 ===

OLLAMA_BASE_URL = "http://localhost:11434"

def check_ollama_status():
    """Ollama 서버 상태 확인"""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get("models", [])
            return True, models
        return False, []
    except requests.exceptions.ConnectionError:
        return False, []
    except Exception as e:
        return False, []

def ollama_generate(model, prompt, temperature=0.1):
    """Ollama API로 직접 텍스트 생성"""
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "temperature": temperature,
                "stream": False
            },
            timeout=30
        )
        if response.status_code == 200:
            return response.json().get("response", "")
        return f"Error: {response.status_code}"
    except Exception as e:
        return f"Error: {e}"

# --- 연결 테스트 ---
print("=" * 60)
print("  Ollama 연결 테스트")
print("=" * 60)

is_running, models = check_ollama_status()

if is_running:
    print(f"\nOllama 상태: 실행 중")
    print(f"사용 가능 모델: {len(models)}개")
    for m in models:
        name = m.get('name', 'unknown')
        size = m.get('size', 0) / (1024**3)  # GB
        print(f"  - {name} ({size:.1f} GB)")
    
    # 첫 번째 모델로 테스트 생성
    if models:
        test_model = models[0]['name']
        print(f"\n테스트 생성 (모델: {test_model}):")
        result = ollama_generate(test_model, "SELECT 1+1의 결과는?")
        print(f"  응답: {result[:200]}")
else:
    print(f"\nOllama 상태: 미실행")
    print(f"  -> 'ollama serve' 명령으로 Ollama를 먼저 시작하세요")
    print(f"  -> 또는 Phase 8 실습 환경을 확인하세요")
    print(f"")
    print(f"이 노트북은 Ollama 없이도 학습 가능합니다.")
    print(f"SQL Agent는 시뮬레이션 모드로 동작합니다.")

---
## 5. 간단한 SQL 에이전트 구축

실제 동작하는 SQL Agent를 구현한다.  
파이프라인: **자연어 질문 -> SQL 생성 -> sqlite3 실행 -> 결과 해석**

| 단계 | 담당 | 핵심 기술 |
|------|------|----------|
| 질문 분석 | LLM (Planning) | 의도 파악, 필요 테이블 식별 |
| SQL 생성 | LLM (파인튜닝 모델) | Text-to-SQL 변환 |
| SQL 실행 | Tool (sqlite3) | 데이터베이스 직접 쿼리 |
| 결과 해석 | LLM (Action) | 쿼리 결과를 자연어로 변환 |

In [ ]:
# === SQL Agent 구현 ===

class SQLAgent:
    """자연어 질문을 SQL로 변환하여 실행하는 에이전트"""
    
    def __init__(self, db_path=":memory:", use_ollama=False, model_name=None):
        self.conn = sqlite3.connect(db_path)
        self.cursor = self.conn.cursor()
        self.use_ollama = use_ollama
        self.model_name = model_name
        self.history = []  # Memory: 대화 히스토리
        self.schema_info = ""  # 스키마 정보 캐시
    
    def setup_database(self):
        """샘플 데이터베이스 생성"""
        # 테이블 생성
        self.cursor.executescript('''
            CREATE TABLE IF NOT EXISTS employees (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                department TEXT NOT NULL,
                salary INTEGER NOT NULL,
                hire_date TEXT NOT NULL
            );
            
            CREATE TABLE IF NOT EXISTS departments (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                budget INTEGER NOT NULL,
                manager TEXT NOT NULL
            );
        ''')
        
        # 샘플 데이터 삽입
        employees = [
            (1, '김철수', 'Engineering', 95000, '2022-03-15'),
            (2, '이영희', 'Engineering', 88000, '2023-01-10'),
            (3, '박민수', 'Marketing', 75000, '2021-07-20'),
            (4, '정수진', 'Marketing', 82000, '2022-11-05'),
            (5, '한지은', 'Sales', 70000, '2023-06-01'),
            (6, '최동현', 'Sales', 68000, '2022-09-12'),
            (7, '윤서연', 'Engineering', 105000, '2020-01-15'),
            (8, '강태우', 'HR', 72000, '2023-04-20'),
            (9, '임소연', 'HR', 78000, '2021-08-30'),
            (10, '조현우', 'Engineering', 92000, '2022-06-18'),
        ]
        
        departments = [
            (1, 'Engineering', 500000, '윤서연'),
            (2, 'Marketing', 300000, '정수진'),
            (3, 'Sales', 250000, '한지은'),
            (4, 'HR', 200000, '임소연'),
        ]
        
        self.cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)', employees)
        self.cursor.executemany('INSERT OR REPLACE INTO departments VALUES (?,?,?,?)', departments)
        self.conn.commit()
        
        self.schema_info = (
            "employees(id INT, name TEXT, department TEXT, salary INT, hire_date TEXT)\n"
            "departments(id INT, name TEXT, budget INT, manager TEXT)"
        )
        print("데이터베이스 초기화 완료")
        print(f"  employees: 10행, departments: 4행")
        print(f"  스키마: {self.schema_info}")
    
    def generate_sql(self, question):
        """Planning + SQL 생성: 자연어 질문을 SQL로 변환"""
        if self.use_ollama:
            prompt = (
                f"스키마: {self.schema_info}\n"
                f"질문: {question}\n"
                f"위 스키마를 참고하여 SQL 쿼리만 작성하세요. 설명 없이 SQL만 출력하세요."
            )
            return ollama_generate(self.model_name, prompt)
        else:
            # 시뮬레이션 모드: 패턴 매칭으로 SQL 생성
            q = question.lower()
            if '평균' in q and '급여' in q and '부서' in q:
                return "SELECT department, ROUND(AVG(salary), 0) as avg_salary FROM employees GROUP BY department ORDER BY avg_salary DESC"
            elif '가장 많' in q and '부서' in q:
                return "SELECT department, COUNT(*) as cnt FROM employees GROUP BY department ORDER BY cnt DESC LIMIT 1"
            elif '최고' in q and '급여' in q:
                return "SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 1"
            elif '전체' in q and '직원' in q:
                return "SELECT COUNT(*) as total FROM employees"
            elif '입사' in q and '최근' in q:
                return "SELECT name, hire_date FROM employees ORDER BY hire_date DESC LIMIT 3"
            elif '예산' in q and '부서' in q:
                return "SELECT name, budget FROM departments ORDER BY budget DESC"
            else:
                return "SELECT * FROM employees LIMIT 5"
    
    def execute_sql(self, sql):
        """Tool Use: SQL 실행"""
        try:
            self.cursor.execute(sql)
            columns = [desc[0] for desc in self.cursor.description]
            rows = self.cursor.fetchall()
            results = [dict(zip(columns, row)) for row in rows]
            return {"success": True, "data": results, "columns": columns}
        except Exception as e:
            return {"success": False, "error": str(e)}
    
    def interpret_result(self, question, sql, result):
        """Action: 결과를 자연어로 해석"""
        if not result["success"]:
            return f"SQL 실행 오류: {result['error']}"
        
        data = result["data"]
        if not data:
            return "조회 결과가 없습니다."
        
        # 결과 포맷팅
        lines = []
        for row in data:
            parts = [f"{k}: {v}" for k, v in row.items()]
            lines.append(", ".join(parts))
        
        return "\n".join(lines)
    
    def ask(self, question):
        """에이전트 실행: 질문 -> SQL 생성 -> 실행 -> 해석"""
        print(f"\n{'='*50}")
        print(f"  [질문] {question}")
        print(f"{'='*50}")
        
        # Step 1: Planning + SQL 생성
        sql = self.generate_sql(question)
        print(f"  [SQL 생성] {sql}")
        
        # Step 2: Tool Use - SQL 실행
        result = self.execute_sql(sql)
        print(f"  [실행 결과] 성공={result['success']}, 행수={len(result.get('data', []))}")
        
        # Step 3: Action - 결과 해석
        interpretation = self.interpret_result(question, sql, result)
        print(f"  [응답] {interpretation}")
        
        # Memory: 히스토리 저장
        self.history.append({
            "question": question,
            "sql": sql,
            "result": result,
            "interpretation": interpretation
        })
        
        return interpretation

# --- 에이전트 초기화 ---
agent = SQLAgent(use_ollama=False)  # 시뮬레이션 모드
agent.setup_database()
print("\nSQL Agent 준비 완료 (시뮬레이션 모드)")

In [ ]:
# === SQL Agent 테스트: 3개 질문으로 End-to-End 테스트 ===

test_questions = [
    "부서별 평균 급여를 알려줘",
    "최고 급여를 받는 직원은 누구야?",
    "가장 최근에 입사한 직원 3명은?",
]

print("\n" + "#" * 60)
print("  SQL Agent End-to-End 테스트")
print("#" * 60)

for q in test_questions:
    agent.ask(q)

# 히스토리 요약
print(f"\n\n{'='*50}")
print(f"  Agent Memory: 대화 히스토리 ({len(agent.history)}건)")
print(f"{'='*50}")
for i, h in enumerate(agent.history, 1):
    print(f"  {i}. Q: {h['question']}")
    print(f"     SQL: {h['sql'][:60]}...")
    print(f"     성공: {h['result']['success']}")

---
## 6. 에이전트 품질: 범용 vs 파인튜닝

에이전트의 **뇌(LLM)**를 바꾸면 어떤 차이가 생기는가?

| 비교 항목 | 범용 LLM | 파인튜닝 모델 |
|----------|----------|-------------|
| SQL 정확도 | 60~70% | 85~95% |
| 도메인 스키마 이해 | 프롬프트 의존 | 학습에 내재 |
| 응답 형식 일관성 | 불안정 | 안정적 |
| 추론 비용 | API 과금 | 로컬 무료 (Ollama) |

In [ ]:
# === 범용 모델 에이전트 vs SQL-파인튜닝 에이전트 시뮬레이션 ===

np.random.seed(42)

# 난이도별 질문 카테고리
difficulty_levels = ['Easy\n(단순 조회)', 'Medium\n(집계/그룹)', 'Hard\n(서브쿼리)', 'Expert\n(복합 JOIN)']

# 정확도 시뮬레이션 (각 난이도별 50개 질문 테스트 가정)
general_accuracy = [78, 62, 45, 32]     # 범용 LLM: 난이도 올라갈수록 급격히 하락
finetuned_accuracy = [95, 91, 83, 72]   # 파인튜닝: 높은 난이도에서도 견고

# 응답 시간 시뮬레이션 (ms)
general_time = [850, 1200, 1800, 2500]    # API 호출 (네트워크 지연 포함)
finetuned_time = [120, 180, 350, 520]     # 로컬 Ollama (네트워크 지연 없음)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Panel 1: 난이도별 정확도 ---
x = np.arange(len(difficulty_levels))
width = 0.35

bars1 = axes[0].bar(x - width/2, general_accuracy, width,
                     label='범용 LLM Agent', color='#c4c4c4', edgecolor='black')
bars2 = axes[0].bar(x + width/2, finetuned_accuracy, width,
                     label='SQL 파인튜닝 Agent', color='#51cf66', edgecolor='black')

axes[0].set_ylabel('SQL 정확도 (%)', fontsize=12)
axes[0].set_title('난이도별 SQL 생성 정확도', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(difficulty_levels, fontsize=10)
axes[0].set_ylim(0, 105)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=9, fontweight='bold')
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height()}%', ha='center', fontsize=9, fontweight='bold')

# 차이 강조 화살표
for i in range(len(difficulty_levels)):
    diff = finetuned_accuracy[i] - general_accuracy[i]
    mid_x = x[i]
    axes[0].annotate(f'+{diff}%p', xy=(mid_x, max(general_accuracy[i], finetuned_accuracy[i]) + 6),
                     ha='center', fontsize=8, color='#2d8a4e', fontweight='bold')

# --- Panel 2: 응답 시간 ---
bars3 = axes[1].bar(x - width/2, general_time, width,
                     label='범용 LLM (API)', color='#ff6b6b', edgecolor='black')
bars4 = axes[1].bar(x + width/2, finetuned_time, width,
                     label='파인튜닝 (Ollama 로컬)', color='#339af0', edgecolor='black')

axes[1].set_ylabel('응답 시간 (ms)', fontsize=12)
axes[1].set_title('난이도별 응답 시간', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(difficulty_levels, fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

for bar in bars3:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height())}ms', ha='center', fontsize=8)
for bar in bars4:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height())}ms', ha='center', fontsize=8)

plt.suptitle('범용 LLM Agent vs SQL 파인튜닝 Agent: 성능 비교',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# 수치 요약
print("\n" + "=" * 60)
print("  성능 차이 요약")
print("=" * 60)
avg_gen_acc = np.mean(general_accuracy)
avg_ft_acc = np.mean(finetuned_accuracy)
avg_gen_time = np.mean(general_time)
avg_ft_time = np.mean(finetuned_time)

print(f"  평균 정확도  | 범용: {avg_gen_acc:.1f}% | 파인튜닝: {avg_ft_acc:.1f}% | 차이: +{avg_ft_acc-avg_gen_acc:.1f}%p")
print(f"  평균 응답시간 | 범용: {avg_gen_time:.0f}ms | 파인튜닝: {avg_ft_time:.0f}ms | {avg_gen_time/avg_ft_time:.1f}x 빠름")
print(f"  비용        | 범용: API 과금 (토큰당) | 파인튜닝: 로컬 무료")
print(f"\n  난이도가 높을수록 파인튜닝 모델의 우위가 커진다.")
print(f"  Expert 난이도에서 정확도 차이: {finetuned_accuracy[-1] - general_accuracy[-1]}%p")

---
## 7. 핵심 정리

| 개념 | 핵심 |
|------|------|
| **LLM Agent** | Planning + Tool Use + Memory + Action |
| **Function Calling** | 모델이 '언제, 어떤 도구를 호출할지' 스스로 결정하는 형식 |
| **LangChain** | LLM + Prompt + Chain + Tool + Memory 프레임워크 |
| **Ollama 연결** | `ChatOllama` 또는 `requests`로 로컬 모델을 에이전트에 장착 |
| **파인튜닝 이점** | 도메인 정확도 +30%p, 응답시간 5x 빠름, 비용 무료 |

### 핵심 원칙

> **에이전트의 '뇌'를 바꾸면 행동이 바뀐다.**  
> 범용 LLM은 범용 에이전트를 만들고, 도메인 특화 파인튜닝 모델은 도메인 전문가 에이전트를 만든다.  
> Phase 4에서 만든 모델이 여기서 비로소 '일하는 시스템'이 된다.

In [ ]:
# === Phase 10-01 체크포인트 ===

print("=" * 60)
print("  Phase 10-01 체크포인트")
print("=" * 60)

checkpoints = [
    ("LLM Agent 4대 요소",
     "Planning(작업 분해), Tool Use(도구 호출), Memory(상태 유지), Action(실행)"),
    ("Function Calling 데이터 형식",
     "user -> assistant(function_call) -> function(결과) -> assistant(해석)"),
    ("LangChain 핵심 컴포넌트",
     "LLM(추론), Prompt(입력), Chain(파이프라인), Tool(도구), Memory(상태)"),
    ("Ollama + LangChain 연결",
     "ChatOllama(model, base_url) | prompt | llm | parser -> LCEL 체인"),
    ("파인튜닝 모델 이점",
     "도메인 정확도 85-95%, 형식 일관성, 로컬 추론(무료/저지연)"),
]

for i, (topic, answer) in enumerate(checkpoints, 1):
    print(f"\n  {i}. {topic}")
    print(f"     -> {answer}")

print(f"\n{'='*60}")
print(f"  Phase 10-01 완료")
print(f"  다음: 02_에이전트_도구_확장.ipynb")
print(f"{'='*60}")